
# Roxy notebook example: Functionally relevant residue content descriptors

This notebook is a **reference implementation example** for the **functionally relevant residue content descriptor family** in Roxy.

These descriptors summarize the content of residues that are frequently associated with **specific biochemical roles**, such as:

- catalysis
- redox chemistry
- aromatic interactions
- nucleophilicity
- hydrogen-bonding potential
- potential disorder-related functionality
- sulfur chemistry
- phosphorylation-prone content proxies

They are useful because many proteins and peptides can be partially characterized by the **enrichment or depletion of residues with known biochemical relevance**, even without structural information.

## Covered outputs

This notebook implements examples such as:

- catalytic-like residue fractions
- nucleophilic residue fractions
- sulfur-containing residue fractions
- aromatic / pi-interaction residue fractions
- hydroxyl-bearing residue fractions
- amide-containing residue fractions
- glycine/proline content proxies
- basic / acidic catalytic-support proxies
- histidine/cysteine/serine/aspartate/glutamate content
- phosphorylation-prone residue burden proxy
- redox-sensitive residue burden proxy
- hydrogen-bond donor / acceptor burden proxies
- class-style implementation for later migration into Roxy


In [1]:

import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "func_1",
            "func_2",
            "func_3",
            "func_4",
            "func_5",
            "func_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,func_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,func_2,GGGGGGGGGGGGGGG,B
2,func_3,KRRKRRKRRKRRDDDDEE,A
3,func_4,ACDEFGHIKLMNPQRSTVWY,B
4,func_5,PPPPGSSSSSTTTTNNQQQ,A
5,func_6,MSTNPKPQRITLKDGNKVELV,B


## Constants

In [3]:

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

FUNCTIONAL_GROUPS = {
    "catalytic_core_like": set("HSCDEKRY"),
    "catalytic_nucleophilic": set("SCYTK"),
    "acid_base_active": set("HDEKRY"),
    "redox_sensitive": set("CMYH"),
    "sulfur_containing": set("CM"),
    "aromatic_pi": set("FWYH"),
    "hydroxyl_bearing": set("STY"),
    "amide_containing": set("NQ"),
    "flexibility_related": set("GP"),
    "basic_functional": set("KRH"),
    "acidic_functional": set("DE"),
    "phosphorylation_prone_proxy": set("STY"),
    "metal_binding_like": set("HCDE"),
    "nucleic_acid_binding_like": set("KRH"),
    "interface_like_aromatic_basic": set("FWYHKR"),
    "small_reactive": set("GACS"),
}

AA_SINGLETS = {
    "H": set("H"),
    "C": set("C"),
    "S": set("S"),
    "D": set("D"),
    "E": set("E"),
    "K": set("K"),
    "R": set("R"),
    "Y": set("Y"),
    "W": set("W"),
    "G": set("G"),
    "P": set("P"),
    "T": set("T"),
    "N": set("N"),
    "Q": set("Q"),
    "M": set("M"),
}

HBOND_DONORS = {
    "A": 0, "C": 0, "D": 0, "E": 0, "F": 0,
    "G": 0, "H": 1, "I": 0, "K": 1, "L": 0,
    "M": 0, "N": 1, "P": 0, "Q": 1, "R": 1,
    "S": 1, "T": 1, "V": 0, "W": 1, "Y": 1,
}

HBOND_ACCEPTORS = {
    "A": 0, "C": 1, "D": 2, "E": 2, "F": 0,
    "G": 0, "H": 1, "I": 0, "K": 0, "L": 0,
    "M": 1, "N": 1, "P": 0, "Q": 1, "R": 0,
    "S": 1, "T": 1, "V": 0, "W": 0, "Y": 1,
}


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA])


def fraction_from_group(seq: str, aa_group) -> float:
    if len(seq) == 0:
        return np.nan
    return sum(aa in aa_group for aa in seq) / len(seq)


def count_from_group(seq: str, aa_group) -> int:
    return sum(aa in aa_group for aa in seq)


def safe_ratio(a: float, b: float) -> float:
    if b == 0:
        return np.nan
    return a / b


## Core descriptor function

In [5]:

def functional_residue_content_descriptors(seq: str) -> dict:
    seq = clean_sequence(seq)

    out = {
        "func_length": len(seq),
        "func_valid_residue_count": len(seq),
    }

    if len(seq) == 0:
        return out

    # Group-based descriptors
    for name, group in FUNCTIONAL_GROUPS.items():
        out[f"func_{name}_count"] = count_from_group(seq, group)
        out[f"func_{name}_fraction"] = fraction_from_group(seq, group)

    # Single-residue descriptors
    for aa, group in AA_SINGLETS.items():
        out[f"func_{aa}_count"] = count_from_group(seq, group)
        out[f"func_{aa}_fraction"] = fraction_from_group(seq, group)

    # Composite functional proxies
    out["func_his_cys_ser_fraction"] = fraction_from_group(seq, set("HCS"))
    out["func_asp_glu_his_fraction"] = fraction_from_group(seq, set("DEH"))
    out["func_lys_arg_his_fraction"] = fraction_from_group(seq, set("KRH"))
    out["func_triad_like_fraction"] = fraction_from_group(seq, set("HSD"))
    out["func_redox_phospho_overlap_fraction"] = fraction_from_group(seq, set("CYT"))
    out["func_aromatic_basic_fraction"] = fraction_from_group(seq, set("FWYHKR"))
    out["func_gly_pro_fraction"] = fraction_from_group(seq, set("GP"))

    # Ratios
    out["func_basic_acidic_ratio"] = safe_ratio(
        count_from_group(seq, set("KRH")),
        count_from_group(seq, set("DE")),
    )
    out["func_aromatic_sulfur_ratio"] = safe_ratio(
        count_from_group(seq, set("FWYH")),
        count_from_group(seq, set("CM")),
    )
    out["func_hydroxyl_amide_ratio"] = safe_ratio(
        count_from_group(seq, set("STY")),
        count_from_group(seq, set("NQ")),
    )
    out["func_cys_met_ratio"] = safe_ratio(
        count_from_group(seq, set("C")),
        count_from_group(seq, set("M")),
    )

    # Hydrogen-bond proxies
    out["func_hbond_donors_per_residue"] = sum(HBOND_DONORS[aa] for aa in seq) / len(seq)
    out["func_hbond_acceptors_per_residue"] = sum(HBOND_ACCEPTORS[aa] for aa in seq) / len(seq)
    out["func_hbond_balance"] = out["func_hbond_donors_per_residue"] - out["func_hbond_acceptors_per_residue"]

    return out


## Functional usage on one sequence

In [6]:

example = functional_residue_content_descriptors(df_demo.loc[0, "sequence"])
list(example.items())[:20]


[('func_length', 24),
 ('func_valid_residue_count', 24),
 ('func_catalytic_core_like_count', 9),
 ('func_catalytic_core_like_fraction', 0.375),
 ('func_catalytic_nucleophilic_count', 7),
 ('func_catalytic_nucleophilic_fraction', 0.2916666666666667),
 ('func_acid_base_active_count', 5),
 ('func_acid_base_active_fraction', 0.20833333333333334),
 ('func_redox_sensitive_count', 2),
 ('func_redox_sensitive_fraction', 0.08333333333333333),
 ('func_sulfur_containing_count', 1),
 ('func_sulfur_containing_fraction', 0.041666666666666664),
 ('func_aromatic_pi_count', 6),
 ('func_aromatic_pi_fraction', 0.25),
 ('func_hydroxyl_bearing_count', 6),
 ('func_hydroxyl_bearing_fraction', 0.25),
 ('func_amide_containing_count', 0),
 ('func_amide_containing_fraction', 0.0),
 ('func_flexibility_related_count', 1),
 ('func_flexibility_related_fraction', 0.041666666666666664)]

## Apply functional residue content descriptors to the full dataset

In [7]:

df_func = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(functional_residue_content_descriptors).apply(pd.Series),
    ],
    axis=1,
)

df_func.head()


,sequence_id,sequence,label,func_length,func_valid_residue_count,func_catalytic_core_like_count,func_catalytic_core_like_fraction,func_catalytic_nucleophilic_count,func_catalytic_nucleophilic_fraction,func_acid_base_active_count,...,func_redox_phospho_overlap_fraction,func_aromatic_basic_fraction,func_gly_pro_fraction,func_basic_acidic_ratio,func_aromatic_sulfur_ratio,func_hydroxyl_amide_ratio,func_cys_met_ratio,func_hbond_donors_per_residue,func_hbond_acceptors_per_residue,func_hbond_balance
0,func_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,9.0,0.375000,7.0,0.291667,5.0,...,0.083333,0.416667,0.041667,NaN,6.0,NaN,0.0,0.458333,0.291667,0.166667
1,func_2,GGGGGGGGGGGGGGG,B,15.0,15.0,0.0,0.000000,0.0,0.000000,0.0,...,0.000000,0.000000,1.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000
2,func_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,18.0,1.000000,4.0,0.222222,18.0,...,0.000000,0.666667,0.000000,2.0,NaN,NaN,NaN,0.666667,0.666667,0.000000
3,func_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,8.0,0.400000,5.0,0.250000,6.0,...,0.150000,0.300000,0.100000,1.5,2.0,1.5,1.0,0.450000,0.600000,-0.150000
4,func_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,5.0,0.263158,9.0,0.473684,0.0,...,0.210526,0.000000,0.263158,NaN,NaN,1.8,NaN,0.736842,0.736842,0.000000


## Inspect descriptor columns

In [8]:

func_cols = [c for c in df_func.columns if c.startswith("func_") and c not in {"func_length", "func_valid_residue_count"}]
len(func_cols), func_cols[:18]


(76,
 ['func_catalytic_core_like_count',
  'func_catalytic_core_like_fraction',
  'func_catalytic_nucleophilic_count',
  'func_catalytic_nucleophilic_fraction',
  'func_acid_base_active_count',
  'func_acid_base_active_fraction',
  'func_redox_sensitive_count',
  'func_redox_sensitive_fraction',
  'func_sulfur_containing_count',
  'func_sulfur_containing_fraction',
  'func_aromatic_pi_count',
  'func_aromatic_pi_fraction',
  'func_hydroxyl_bearing_count',
  'func_hydroxyl_bearing_fraction',
  'func_amide_containing_count',
  'func_amide_containing_fraction',
  'func_flexibility_related_count',
  'func_flexibility_related_fraction'])

In [9]:

df_func[
    [
        "sequence_id",
        "func_catalytic_core_like_fraction",
        "func_redox_sensitive_fraction",
        "func_phosphorylation_prone_proxy_fraction",
        "func_H_fraction",
        "func_C_fraction",
        "func_basic_acidic_ratio",
        "func_hbond_donors_per_residue",
    ]
]


,sequence_id,func_catalytic_core_like_fraction,func_redox_sensitive_fraction,func_phosphorylation_prone_proxy_fraction,func_H_fraction,func_C_fraction,func_basic_acidic_ratio,func_hbond_donors_per_residue
0,func_1,0.375000,0.083333,0.250000,0.00,0.00,NaN,0.458333
1,func_2,0.000000,0.000000,0.000000,0.00,0.00,NaN,0.000000
2,func_3,1.000000,0.000000,0.000000,0.00,0.00,2.0,0.666667
3,func_4,0.400000,0.200000,0.150000,0.05,0.05,1.5,0.450000
4,func_5,0.263158,0.000000,0.473684,0.00,0.00,NaN,0.736842
5,func_6,0.333333,0.047619,0.142857,0.00,0.00,2.0,0.476190


## Dataset-level summary

In [10]:

func_summary = (
    df_func[func_cols]
    .mean(axis=0, numeric_only=True)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

func_summary.head(15)


,descriptor,mean_value
0,func_catalytic_core_like_count,7.833333
1,func_acid_base_active_count,5.833333
2,func_small_reactive_count,5.500000
3,func_interface_like_aromatic_basic_count,5.333333
4,func_catalytic_nucleophilic_count,5.166667
5,func_flexibility_related_count,4.333333
6,func_basic_functional_count,3.833333
7,func_nucleic_acid_binding_like_count,3.833333
8,func_hydroxyl_bearing_count,3.500000
9,func_phosphorylation_prone_proxy_count,3.500000


## Sanity checks

In [11]:

assert "func_catalytic_core_like_fraction" in df_func.columns
assert "func_redox_sensitive_fraction" in df_func.columns
assert "func_phosphorylation_prone_proxy_fraction" in df_func.columns
assert "func_H_fraction" in df_func.columns
assert "func_basic_acidic_ratio" in df_func.columns
assert "func_hbond_donors_per_residue" in df_func.columns
assert df_func["func_length"].min() > 0

print(f"Number of functionally relevant residue descriptor columns: {len(func_cols)}")
print("Functional residue content descriptor checks passed.")


Number of functionally relevant residue descriptor columns: 76
Functional residue content descriptor checks passed.


## Class-style implementation closer to the real package

In [12]:

class FunctionalResidueContentDescriptors:
    """Example class-style implementation for later migration into Roxy."""

    def transform_sequence(self, seq: str) -> dict:
        return functional_residue_content_descriptors(seq)

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


func_transformer = FunctionalResidueContentDescriptors()
func_matrix = func_transformer.transform(df_demo["sequence"].tolist())
func_matrix.head()


,func_length,func_valid_residue_count,func_catalytic_core_like_count,func_catalytic_core_like_fraction,func_catalytic_nucleophilic_count,func_catalytic_nucleophilic_fraction,func_acid_base_active_count,func_acid_base_active_fraction,func_redox_sensitive_count,func_redox_sensitive_fraction,...,func_redox_phospho_overlap_fraction,func_aromatic_basic_fraction,func_gly_pro_fraction,func_basic_acidic_ratio,func_aromatic_sulfur_ratio,func_hydroxyl_amide_ratio,func_cys_met_ratio,func_hbond_donors_per_residue,func_hbond_acceptors_per_residue,func_hbond_balance
0,24,24,9,0.375000,7,0.291667,5,0.208333,2,0.083333,...,0.083333,0.416667,0.041667,NaN,6.0,NaN,0.0,0.458333,0.291667,0.166667
1,15,15,0,0.000000,0,0.000000,0,0.000000,0,0.000000,...,0.000000,0.000000,1.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000
2,18,18,18,1.000000,4,0.222222,18,1.000000,0,0.000000,...,0.000000,0.666667,0.000000,2.0,NaN,NaN,NaN,0.666667,0.666667,0.000000
3,20,20,8,0.400000,5,0.250000,6,0.300000,4,0.200000,...,0.150000,0.300000,0.100000,1.5,2.0,1.5,1.0,0.450000,0.600000,-0.150000
4,19,19,5,0.263158,9,0.473684,0,0.000000,0,0.000000,...,0.210526,0.000000,0.263158,NaN,NaN,1.8,NaN,0.736842,0.736842,0.000000


## Merge transformer output back to the dataset

In [13]:

df_func_class = pd.concat([df_demo, func_matrix], axis=1)
df_func_class.head()


,sequence_id,sequence,label,func_length,func_valid_residue_count,func_catalytic_core_like_count,func_catalytic_core_like_fraction,func_catalytic_nucleophilic_count,func_catalytic_nucleophilic_fraction,func_acid_base_active_count,...,func_redox_phospho_overlap_fraction,func_aromatic_basic_fraction,func_gly_pro_fraction,func_basic_acidic_ratio,func_aromatic_sulfur_ratio,func_hydroxyl_amide_ratio,func_cys_met_ratio,func_hbond_donors_per_residue,func_hbond_acceptors_per_residue,func_hbond_balance
0,func_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,9,0.375000,7,0.291667,5,...,0.083333,0.416667,0.041667,NaN,6.0,NaN,0.0,0.458333,0.291667,0.166667
1,func_2,GGGGGGGGGGGGGGG,B,15,15,0,0.000000,0,0.000000,0,...,0.000000,0.000000,1.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000
2,func_3,KRRKRRKRRKRRDDDDEE,A,18,18,18,1.000000,4,0.222222,18,...,0.000000,0.666667,0.000000,2.0,NaN,NaN,NaN,0.666667,0.666667,0.000000
3,func_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,8,0.400000,5,0.250000,6,...,0.150000,0.300000,0.100000,1.5,2.0,1.5,1.0,0.450000,0.600000,-0.150000
4,func_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,5,0.263158,9,0.473684,0,...,0.210526,0.000000,0.263158,NaN,NaN,1.8,NaN,0.736842,0.736842,0.000000



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move functional residue libraries into `roxy/core/constants.py`
- move helper logic into `roxy/sequence/functional.py`
- expose a class such as `FunctionalResidueContentDescriptors`
- allow configurable:
  - functional residue group libraries
  - residue-level outputs
  - ratio and proxy subsets
- add tests for:
  - empty sequences
  - highly catalytic-like enriched sequences
  - sulfur-rich sequences
  - phosphorylation-prone residue enrichment
  - lower-case input
  - invalid characters removed during cleaning


## Optional export

In [14]:
# df_func.to_csv("demo_functional_residue_content_descriptors.csv", index=False)
